# Norwegian Loan Calculator
## Annuity vs Serial Loan Comparison with Tax Deductions & Rental Income

This notebook compares **annuity** and **serial (linear)** loans under Norwegian tax law:

| Rule | Detail |
|------|--------|
| **Rentefradrag** | 22% tax deduction on interest payments |
| **Rental (own home, ≥50% owner-occupied)** | Tax-free |
| **Rental (separate unit / full property)** | 22% capital income tax (minus maintenance) |
| **Short-term rental (<30 days)** | First NOK 10,000 tax-free, then 85% × 22% |

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from norwegian_loan_calculator import (
    LoanParameters,
    RentalParameters,
    compare_loans,
    print_comparison,
    print_annual_schedule,
    format_nok,
)

plt.rcParams.update({"figure.figsize": (14, 5), "figure.dpi": 100})

def nok_formatter(x, _):
    return f"{x:,.0f}".replace(",", " ")

## 1. Define Loan Parameters

Adjust the values below to match your scenario.

In [ ]:
loan = LoanParameters(
    principal=3_500_000,           # NOK – loan amount
    annual_interest_rate=0.045,    # 4.5%
    term_years=25,                 # 25 years
    extra_fees_per_month=50,       # NOK – termingebyr
)

# Rental income (set monthly_rental_income=0 to disable)
rental = RentalParameters(
    monthly_rental_income=8_000,   # NOK/month
    is_short_term=False,           # Long-term rental
    owner_occupies_half=True,      # Owner lives in ≥50% → tax-free rental
    annual_maintenance_cost=0,     # Deductible maintenance (only for taxable rental)
)

annuity, serial = compare_loans(loan, rental)
print_comparison(annuity, serial)

## 2. Monthly Payment Over Time

The key difference: annuity has **constant payments**, serial starts **high and decreases**.

In [ ]:
months = [mp.month for mp in annuity.monthly_payments]
ann_payments = [mp.total_payment for mp in annuity.monthly_payments]
ser_payments = [mp.total_payment for mp in serial.monthly_payments]

fig, ax = plt.subplots()
ax.plot(months, ann_payments, label="Annuity", linewidth=1.5)
ax.plot(months, ser_payments, label="Serial", linewidth=1.5)
ax.set_xlabel("Month")
ax.set_ylabel("Monthly payment (NOK)")
ax.set_title("Monthly Payment – Annuity vs Serial")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(nok_formatter))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Interest vs Principal Split (Stacked Area)

Annual breakdown showing how much goes to interest vs principal repayment.

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True)

for ax, summary, title in zip(axes, [annuity, serial], ["Annuity", "Serial"]):
    years = [a.year for a in summary.annual_summaries]
    interest = [a.total_interest for a in summary.annual_summaries]
    principal = [a.total_principal for a in summary.annual_summaries]

    ax.bar(years, principal, label="Principal", color="#2196F3")
    ax.bar(years, interest, bottom=principal, label="Interest", color="#FF7043")
    ax.set_title(f"{title} Loan")
    ax.set_xlabel("Year")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(nok_formatter))
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3, axis="y")

axes[0].set_ylabel("Annual payment (NOK)")
fig.suptitle("Principal vs Interest Breakdown", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Remaining Balance Over Time

In [ ]:
ann_balance = [mp.remaining_balance for mp in annuity.monthly_payments]
ser_balance = [mp.remaining_balance for mp in serial.monthly_payments]

fig, ax = plt.subplots()
ax.fill_between(months, ann_balance, alpha=0.3, label="Annuity")
ax.fill_between(months, ser_balance, alpha=0.3, label="Serial")
ax.plot(months, ann_balance, linewidth=1.2)
ax.plot(months, ser_balance, linewidth=1.2)
ax.set_xlabel("Month")
ax.set_ylabel("Remaining balance (NOK)")
ax.set_title("Remaining Loan Balance")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(nok_formatter))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Cumulative Cost Comparison (with Tax Deduction)

Shows how the **net cost** (total paid minus 22% interest deduction) accumulates.

In [ ]:
ann_cum_gross = np.cumsum([mp.total_payment for mp in annuity.monthly_payments])
ann_cum_net = np.cumsum([mp.net_payment for mp in annuity.monthly_payments])
ser_cum_gross = np.cumsum([mp.total_payment for mp in serial.monthly_payments])
ser_cum_net = np.cumsum([mp.net_payment for mp in serial.monthly_payments])

fig, ax = plt.subplots()
ax.plot(months, ann_cum_gross, "--", label="Annuity (gross)", alpha=0.5)
ax.plot(months, ann_cum_net, label="Annuity (net after tax ded.)", linewidth=2)
ax.plot(months, ser_cum_gross, "--", label="Serial (gross)", alpha=0.5)
ax.plot(months, ser_cum_net, label="Serial (net after tax ded.)", linewidth=2)
ax.set_xlabel("Month")
ax.set_ylabel("Cumulative cost (NOK)")
ax.set_title("Cumulative Cost – Gross vs Net (after 22% interest deduction)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(nok_formatter))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Annual Net Cost After Tax Deduction & Rental Income

In [ ]:
years = [a.year for a in annuity.annual_summaries]
ann_net_cost = [a.net_cost_after_rental for a in annuity.annual_summaries]
ser_net_cost = [a.net_cost_after_rental for a in serial.annual_summaries]

x = np.arange(len(years))
width = 0.35

fig, ax = plt.subplots()
bars1 = ax.bar(x - width/2, ann_net_cost, width, label="Annuity", color="#2196F3")
bars2 = ax.bar(x + width/2, ser_net_cost, width, label="Serial", color="#FF7043")
ax.set_xlabel("Year")
ax.set_ylabel("Net annual cost (NOK)")
ax.set_title("Annual Net Cost (after tax deduction & rental income)")
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(nok_formatter))
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
ax.axhline(y=0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Total Cost Summary

In [ ]:
print("\n" + "=" * 70)
print("TOTAL COST SUMMARY")
print("=" * 70)
for s in [annuity, serial]:
    print(f"\n--- {s.loan_type} Loan ---")
    print(f"  Total paid:              {format_nok(s.total_paid)}")
    print(f"  Total interest:          {format_nok(s.total_interest)}")
    print(f"  Total fees:              {format_nok(s.total_fees)}")
    print(f"  Tax deduction (22%):    -{format_nok(s.total_tax_deduction)}")
    print(f"  Net cost:                {format_nok(s.total_net_cost)}")
    if s.total_rental_income_gross > 0:
        print(f"  Rental income (net):    -{format_nok(s.total_rental_income_net)}")
        print(f"  Net cost after rental:   {format_nok(s.total_net_cost_after_rental)}")

diff = annuity.total_net_cost_after_rental - serial.total_net_cost_after_rental
print(f"\n  → Serial loan saves you {format_nok(abs(diff))} over {loan.term_years} years.")

## 8. Interest Rate Sensitivity Analysis

How does the total net cost change across different interest rates?

In [ ]:
rates = np.arange(0.02, 0.08, 0.005)
ann_costs = []
ser_costs = []

for r in rates:
    test_loan = LoanParameters(
        principal=loan.principal,
        annual_interest_rate=r,
        term_years=loan.term_years,
        extra_fees_per_month=loan.extra_fees_per_month,
    )
    a, s = compare_loans(test_loan)
    ann_costs.append(a.total_net_cost)
    ser_costs.append(s.total_net_cost)

fig, ax = plt.subplots()
ax.plot(rates * 100, [c / 1e6 for c in ann_costs], "o-", label="Annuity", linewidth=2)
ax.plot(rates * 100, [c / 1e6 for c in ser_costs], "s-", label="Serial", linewidth=2)
ax.set_xlabel("Annual interest rate (%)")
ax.set_ylabel("Total net cost (million NOK)")
ax.set_title("Total Net Cost vs Interest Rate")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Scenario Comparison: Different Rental Strategies

Compare net cost under three rental scenarios:
1. **No rental** – pure housing cost
2. **Long-term rental** (own home, ≥50% occupied) – tax-free
3. **Short-term rental** (Airbnb-style) – partially taxed

In [ ]:
scenarios = {
    "No rental": None,
    "Long-term (tax-free)\nNOK 8,000/mo": RentalParameters(
        monthly_rental_income=8_000,
        is_short_term=False,
        owner_occupies_half=True,
    ),
    "Short-term (taxed)\nNOK 12,000/mo": RentalParameters(
        monthly_rental_income=12_000,
        is_short_term=True,
        owner_occupies_half=False,
    ),
    "Separate unit (taxed)\nNOK 10,000/mo": RentalParameters(
        monthly_rental_income=10_000,
        is_short_term=False,
        owner_occupies_half=False,
        annual_maintenance_cost=15_000,
    ),
}

ann_results = []
ser_results = []
labels = list(scenarios.keys())

for label, r in scenarios.items():
    a, s = compare_loans(loan, r)
    ann_results.append(a.total_net_cost_after_rental)
    ser_results.append(s.total_net_cost_after_rental)

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, [v/1e6 for v in ann_results], width, label="Annuity", color="#2196F3")
bars2 = ax.bar(x + width/2, [v/1e6 for v in ser_results], width, label="Serial", color="#FF7043")

ax.set_ylabel("Total net cost (million NOK)")
ax.set_title("Total Net Cost by Rental Strategy")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.2f}M",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 10. Year-by-Year Schedule

Detailed annual breakdowns for both loan types.

In [ ]:
print_annual_schedule(annuity)
print_annual_schedule(serial)